In [1]:
# Celda C0b - Recarga del entorno del clasificador (sin re-muestrear)
import sys, json, time, re
import pandas as pd
from pathlib import Path

import os
BASE = Path(os.environ.get('TESIS_BASE', '/Users/ppizam/Claude/Master Thesis'))
CLAS = BASE / 'Desarrollo' / 'Metodologia' / 'Clasificador'
sys.path.append(str(BASE / 'Desarrollo' / 'Metodologia' / 'APIS'))
from llm_clients import preguntar

piloto = pd.read_csv(CLAS / 'muestra_piloto_1000.csv', keep_default_na=False, na_values=[''])
LOTE = 20

SYSTEM = (
    'Eres un clasificador de postura direccional en foros financieros de Reddit. '
    'Para cada mensaje numerado, decide la postura del AUTOR respecto del TICKER indicado: '
    '"compra" si expresa postura alcista o intencion de comprar/mantener; '
    '"venta" si expresa postura bajista o intencion de vender/apostar en corto; '
    '"neutral" si no hay postura clara (pregunta, dato informativo, el ticker es incidental, '
    'o la direccion es ambigua). '
    'Considera la jerga del foro (calls, moon, YOLO, diamond hands suelen ser alcistas; '
    'puts, short, drill suelen ser bajistas), el sarcasmo y las negaciones. '
    'Ojo con la mecanica de opciones: vender puts es alcista, vender calls es bajista. '
    'Responde SOLO un arreglo JSON con un objeto por mensaje, sin texto adicional: '
    '[{"i":0,"e":"compra"},{"i":1,"e":"neutral"},...]'
)

def prompt_lote(df):
    lineas = []
    for j, (_, r) in enumerate(df.iterrows()):
        txt = str(r.texto).replace('\n', ' ').strip()[:600]
        lineas.append(f'{j}) TICKER: {r.ticker} | TEXTO: {txt}')
    return 'Clasifica estos mensajes:\n' + '\n'.join(lineas)

def parsear(texto_resp, n_esperado):
    t = texto_resp.strip()
    a, b = t.find('['), t.rfind(']')
    if a == -1 or b == -1:
        return None
    try:
        arr = json.loads(t[a:b + 1])
        et = {int(d['i']): str(d['e']).lower().strip() for d in arr}
        return [et.get(j, 'error') for j in range(n_esperado)]
    except Exception:
        return None

print(f'entorno listo: {len(piloto)} mensajes del piloto en memoria')

entorno listo: 948 mensajes del piloto en memoria


In [ ]:
# Celda C1 - Muestreo de mensajes con mencion del top 50 (piloto del clasificador)
import json, io, random, re
import pandas as pd
import zstandard as zstd
from pathlib import Path

random.seed(42)
import os
BASE = Path(os.environ.get('TESIS_BASE', '/Users/ppizam/Claude/Master Thesis'))
DATA = Path(os.environ.get('REDDIT_DATA', '/Users/ppizam/Library/CloudStorage/GoogleDrive-pizacapital@gmail.com/Other computers/My Mac RRG/data/reddit'))
CLAS = BASE / 'Desarrollo' / 'Metodologia' / 'Clasificador'
CLAS.mkdir(exist_ok=True)

top50 = pd.read_csv(BASE / 'Desarrollo/Metodologia/Matrix/eventos/acciones_principales_top50.csv',
                    keep_default_na=False, na_values=[''])
TICKS = set(top50['ticker'])
RE_CASH = re.compile(r'\$([A-Za-z]{1,5})\b')
RE_TOK = re.compile(r'(?<![A-Za-z$])[A-Z]{2,5}(?![A-Za-z])')

def tickers_en(texto):
    ups = {m.group(1).upper() for m in RE_CASH.finditer(texto)}
    ups |= set(RE_TOK.findall(texto))
    return ups & TICKS

SUBS = ['wallstreetbets', 'stocks']
MESES_POR_ANIO = {a: random.sample(range(1, 13), 2) for a in range(2020, 2026)}
MESES_POR_ANIO[2026] = random.sample(range(1, 7), 2)
CUOTA_POR_ARCHIVO = 40      # candidatos por archivo (luego se balancea)
MAX_LINEAS = 300_000        # tope de lectura por archivo (control de tiempo)
P_CAPTURA = 0.3             # probabilidad de conservar cada mencion encontrada

muestras = []
for anio, meses in MESES_POR_ANIO.items():
    for mes in meses:
        for sub in SUBS:
            for tipo in ('submissions', 'comments'):
                ruta = DATA / f'{anio}' / f'{mes:02d}' / f'{sub}_{tipo}.zst'
                if not ruta.exists() or ruta.stat().st_size == 0:
                    continue
                capturados = 0
                dctx = zstd.ZstdDecompressor(max_window_size=2**31)
                with open(ruta, 'rb') as f, dctx.stream_reader(f) as r:
                    for n, linea in enumerate(io.TextIOWrapper(r, encoding='utf-8',
                                                               errors='replace')):
                        if n >= MAX_LINEAS or capturados >= CUOTA_POR_ARCHIVO:
                            break
                        try:
                            msg = json.loads(linea)
                        except json.JSONDecodeError:
                            continue
                        if tipo == 'submissions':
                            texto = (msg.get('title') or '')
                            cuerpo = (msg.get('selftext') or '')
                            if cuerpo in ('[removed]', '[deleted]'):
                                cuerpo = ''
                            texto_full = (texto + '\n' + cuerpo).strip()[:800]
                        else:
                            texto_full = (msg.get('body') or '')[:800]
                            if texto_full in ('[removed]', '[deleted]', ''):
                                continue
                        tks = tickers_en(texto_full)
                        if not tks or random.random() > P_CAPTURA:
                            continue
                        muestras.append({
                            'anio': anio, 'mes': mes, 'sub': sub, 'tipo': tipo,
                            'id': msg.get('id', ''),
                            'created_utc': msg.get('created_utc', ''),
                            'ticker': sorted(tks)[0],
                            'n_tickers': len(tks),
                            'texto': texto_full,
                        })
                        capturados += 1
                print(f'{anio}-{mes:02d} {sub} {tipo}: {capturados}', flush=True)

mu = pd.DataFrame(muestras)
mu.to_csv(CLAS / 'muestra_cruda.csv', index=False)
print(f'\nmuestra cruda: {len(mu)} mensajes | por anio:')
print(mu.groupby('anio').size().to_string())

In [ ]:
# Celda C2 - Reglas de jerga v2 (negacion + mecanica de opciones) y balanceo a 1,000
ALCISTA = ['buy', 'bought', 'buying', 'long', 'calls', 'call', 'yolo', 'moon',
           'rocket', 'hodl', 'hold', 'diamond hands', 'bull', 'bullish',
           'squeeze', 'to the moon', 'dip', 'undervalued', 'load', 'loading']
BAJISTA = ['sell', 'sold', 'selling', 'short', 'shorting', 'puts', 'put',
           'bear', 'bearish', 'crash', 'dump', 'drill', 'bagholder',
           'collapse', 'tank', 'overvalued', 'bubble']
NEGACIONES = ["don't", 'dont', 'not', 'never', 'no way', "won't", 'wont', "wouldn't"]

def reglas_v2(texto):
    t = ' ' + texto.lower() + ' '
    # mecanica de opciones: selling puts = alcista, selling calls = bajista
    b = len(re.findall(r'\bsell(?:ing)? puts?\b', t))
    s = len(re.findall(r'\bsell(?:ing)? calls?\b', t))
    t_sin = re.sub(r'\bsell(?:ing)? (?:puts?|calls?)\b', ' ', t)
    for w in ALCISTA:
        for m in re.finditer(r'\b' + re.escape(w) + r'\b', t_sin):
            ventana = t_sin[max(0, m.start() - 25):m.start()]
            if any(n in ventana for n in NEGACIONES):
                s += 1        # 'not buying' cuenta bajista
            else:
                b += 1
    for w in BAJISTA:
        for m in re.finditer(r'\b' + re.escape(w) + r'\b', t_sin):
            ventana = t_sin[max(0, m.start() - 25):m.start()]
            if any(n in ventana for n in NEGACIONES):
                b += 1        # 'not selling' cuenta alcista
            else:
                s += 1
    return 'compra' if b > s else ('venta' if s > b else 'neutral'), b, s

res = mu['texto'].apply(reglas_v2)
mu['reglas_v2'] = res.str[0]
mu['b'] = res.str[1]
mu['s'] = res.str[2]
print('distribucion reglas v2 en la muestra cruda:')
print(mu.reglas_v2.value_counts().to_string())

# balanceo: ~1,000 mensajes repartidos por anio y direccion presunta
OBJETIVO = 1000
por_celda = OBJETIVO // (mu.anio.nunique() * 3)
piloto = (mu.groupby(['anio', 'reglas_v2'], group_keys=False)
            .apply(lambda g: g.sample(min(len(g), por_celda), random_state=42)))
piloto = piloto.reset_index(drop=True)
piloto.to_csv(CLAS / 'muestra_piloto_1000.csv', index=False)
print(f'\npiloto balanceado: {len(piloto)} mensajes')
print(piloto.groupby(['anio', 'reglas_v2']).size().unstack(fill_value=0).to_string())

In [ ]:
# Celda C3 - Clasificacion del piloto con los 4 proveedores (lotes de 20, checkpoint)
import sys, json, time
sys.path.append(str(BASE / 'Desarrollo' / 'Metodologia' / 'APIS'))
from llm_clients import preguntar

piloto = pd.read_csv(CLAS / 'muestra_piloto_1000.csv', keep_default_na=False, na_values=[''])
PROVEEDORES = ['deepseek', 'mistral', 'claude', 'kimi']
LOTE = 20

SYSTEM = (
    'Eres un clasificador de postura direccional en foros financieros de Reddit. '
    'Para cada mensaje numerado, decide la postura del AUTOR respecto del TICKER indicado: '
    '"compra" si expresa postura alcista o intencion de comprar/mantener; '
    '"venta" si expresa postura bajista o intencion de vender/apostar en corto; '
    '"neutral" si no hay postura clara (pregunta, dato informativo, el ticker es incidental, '
    'o la direccion es ambigua). '
    'Considera la jerga del foro (calls, moon, YOLO, diamond hands suelen ser alcistas; '
    'puts, short, drill suelen ser bajistas), el sarcasmo y las negaciones. '
    'Ojo con la mecanica de opciones: vender puts es alcista, vender calls es bajista. '
    'Responde SOLO un arreglo JSON con un objeto por mensaje, sin texto adicional: '
    '[{"i":0,"e":"compra"},{"i":1,"e":"neutral"},...]'
)

def prompt_lote(df):
    lineas = []
    for j, (_, r) in enumerate(df.iterrows()):
        txt = str(r.texto).replace('\n', ' ').strip()[:600]
        lineas.append(f'{j}) TICKER: {r.ticker} | TEXTO: {txt}')
    return 'Clasifica estos mensajes:\n' + '\n'.join(lineas)

def parsear(texto_resp, n_esperado):
    t = texto_resp.strip()
    a, b = t.find('['), t.rfind(']')
    if a == -1 or b == -1:
        return None
    try:
        arr = json.loads(t[a:b + 1])
        et = {int(d['i']): str(d['e']).lower().strip() for d in arr}
        return [et.get(j, 'error') for j in range(n_esperado)]
    except Exception:
        return None

resumen = []
for prov in PROVEEDORES:
    archivo = CLAS / f'clasif_llm_{prov}.csv'
    hechas = pd.read_csv(archivo) if archivo.exists() else pd.DataFrame(columns=['idx', 'etiqueta'])
    listos = set(hechas['idx'])
    nuevas, tok_in, tok_out, t0 = [], 0, 0, time.time()
    for ini in range(0, len(piloto), LOTE):
        df = piloto.iloc[ini:ini + LOTE]
        if all(ini + j in listos for j in range(len(df))):
            continue
        r = preguntar(prov, prompt_lote(df), system=SYSTEM, max_tokens=2000)
        etiquetas = None
        if not r.get('error'):
            etiquetas = parsear(r['respuesta'], len(df))
            tok_in += r.get('tokens_entrada') or 0
            tok_out += r.get('tokens_salida') or 0
        if etiquetas is None:
            etiquetas = ['error'] * len(df)
        for j, e in enumerate(etiquetas):
            nuevas.append({'idx': ini + j, 'etiqueta': e})
        if (ini // LOTE) % 5 == 0:
            print(f'{prov}: lote {ini // LOTE + 1}/{(len(piloto) - 1) // LOTE + 1}', flush=True)
            pd.concat([hechas, pd.DataFrame(nuevas)]).to_csv(archivo, index=False)
    todas = pd.concat([hechas, pd.DataFrame(nuevas)]).drop_duplicates('idx').sort_values('idx')
    todas.to_csv(archivo, index=False)
    mins = (time.time() - t0) / 60
    errores = int((todas.etiqueta == 'error').sum())
    resumen.append({'proveedor': prov, 'clasificados': len(todas), 'errores': errores,
                    'tokens_in': tok_in, 'tokens_out': tok_out, 'minutos': round(mins, 1)})
    print(f'--- {prov} terminado: {len(todas)} etiquetas, {errores} errores, {mins:.1f} min')

print()
print(pd.DataFrame(resumen).to_string(index=False))

In [ ]:
# Celda C3b - Reparar lotes fallidos de mistral y kimi
# mistral: reintento simple. kimi: max_tokens amplio para que el razonamiento no trunque el JSON.
REPARAR = {'mistral': dict(max_tokens=2000),
           'kimi': dict(max_tokens=9000)}

for prov, kw in REPARAR.items():
    archivo = CLAS / f'clasif_llm_{prov}.csv'
    tabla = pd.read_csv(archivo)
    malos = sorted(tabla[tabla.etiqueta == 'error'].idx)
    if not malos:
        print(f'{prov}: sin errores que reparar')
        continue
    lotes = sorted({i - (i % LOTE) for i in malos})
    print(f'{prov}: {len(malos)} etiquetas en error, {len(lotes)} lotes a repetir')
    for ini in lotes:
        df = piloto.iloc[ini:ini + LOTE]
        r = preguntar(prov, prompt_lote(df), system=SYSTEM, **kw)
        etiquetas = None if r.get('error') else parsear(r['respuesta'], len(df))
        if etiquetas is None:
            print(f'  lote {ini // LOTE}: sigue fallando'); continue
        for j, e in enumerate(etiquetas):
            tabla.loc[tabla.idx == ini + j, 'etiqueta'] = e
        tabla.to_csv(archivo, index=False)
    quedan = int((tabla.etiqueta == 'error').sum())
    print(f'{prov}: reparado, quedan {quedan} errores')

# Celda C4 v2 - Matriz de acuerdo entre los 4 proveedores y las reglas v2 (corregida)
VALIDAS = {'compra', 'venta', 'neutral'}
comp = piloto[['anio', 'sub', 'tipo', 'ticker', 'texto', 'reglas_v2']].copy()
for prov in PROVEEDORES:
    t = pd.read_csv(CLAS / f'clasif_llm_{prov}.csv').sort_values('idx')
    comp[prov] = t.etiqueta.values
    comp[prov] = comp[prov].where(comp[prov].isin(VALIDAS))   # etiquetas invalidas -> NaN

cols = ['reglas_v2'] + PROVEEDORES
print('distribucion de etiquetas por clasificador:')
print(pd.DataFrame({c: comp[c].value_counts() for c in cols}).fillna(0).astype(int).to_string())

print('\nacuerdo por pares (%):')
acuerdo = pd.DataFrame(index=cols, columns=cols, dtype=float)
for a in cols:
    for b in cols:
        if a == b:
            acuerdo.loc[a, b] = 100.0
            continue
        m = comp[[a, b]].dropna()
        acuerdo.loc[a, b] = round(100 * (m[a] == m[b]).mean(), 1)
print(acuerdo.to_string())

# consenso entre los 4 LLMs
llm = comp[PROVEEDORES]
completos = llm.dropna()
def consenso(fila):
    v = fila.value_counts()
    return v.index[0], int(v.iloc[0])
cv = completos.apply(consenso, axis=1)
comp.loc[completos.index, 'etiqueta_consenso'] = cv.str[0]
comp.loc[completos.index, 'votos'] = cv.str[1]

print(f'\nmensajes con las 4 etiquetas validas: {len(completos)} de {len(comp)}')
print('distribucion de votos del ganador:')
print(comp.votos.value_counts().sort_index().to_string())
print(f"\nconsenso 4/4 (etiqueta de oro): {int((comp.votos == 4).sum())} "
      f"({100 * (comp.votos == 4).mean():.1f}%)")
print('distribucion del consenso 4/4:')
print(comp[comp.votos == 4].etiqueta_consenso.value_counts().to_string())

oro = comp[comp.votos == 4].dropna(subset=['reglas_v2'])
print(f"\nacuerdo reglas v2 vs consenso 4/4: {100 * (oro.reglas_v2 == oro.etiqueta_consenso).mean():.1f}%")

discrepantes = comp[comp.votos <= 2].copy()
comp.to_csv(CLAS / 'piloto_etiquetado_completo.csv', index=False)
discrepantes.to_csv(CLAS / 'discrepancias_para_revision_manual.csv', index=False)
print(f'\ndiscrepantes (votos <= 2, van a revision manual): {len(discrepantes)}')
print('\nejemplos de discrepancia (primeros 5):')
for _, r in discrepantes.head(5).iterrows():
    print(f"  [{r.ticker}] {str(r.texto)[:90]}")
    print(f"    reglas: {r.reglas_v2} | " + ' | '.join(f'{p}: {str(r[p])}' for p in PROVEEDORES))

In [ ]:
exec(open(CLAS / 'celda_C4.py').read())


In [2]:
# Celda C6 - Clasificacion del piloto con OpenAI (gpt-4o-mini), lotes de 20 con checkpoint
prov = 'openai'
archivo = CLAS / f'clasif_llm_{prov}.csv'
hechas = pd.read_csv(archivo) if archivo.exists() else pd.DataFrame(columns=['idx', 'etiqueta'])
listos = set(hechas['idx'])
nuevas, tok_in, tok_out, t0 = [], 0, 0, time.time()

for ini in range(0, len(piloto), LOTE):
    df = piloto.iloc[ini:ini + LOTE]
    if all(ini + j in listos for j in range(len(df))):
        continue
    r = preguntar(prov, prompt_lote(df), system=SYSTEM, max_tokens=2000)
    etiquetas = None
    if not r.get('error'):
        etiquetas = parsear(r['respuesta'], len(df))
        tok_in += r.get('tokens_entrada') or 0
        tok_out += r.get('tokens_salida') or 0
    if etiquetas is None:
        etiquetas = ['error'] * len(df)
    for j, e in enumerate(etiquetas):
        nuevas.append({'idx': ini + j, 'etiqueta': e})
    if (ini // LOTE) % 5 == 0:
        print(f'{prov}: lote {ini // LOTE + 1}/{(len(piloto) - 1) // LOTE + 1}', flush=True)
        pd.concat([hechas, pd.DataFrame(nuevas)]).to_csv(archivo, index=False)

todas = pd.concat([hechas, pd.DataFrame(nuevas)]).drop_duplicates('idx').sort_values('idx')
todas.to_csv(archivo, index=False)
mins = (time.time() - t0) / 60
errores = int((todas.etiqueta == 'error').sum())
costo = (tok_in * 0.15 + tok_out * 0.60) / 1_000_000
print(f'\n--- {prov} terminado: {len(todas)} etiquetas, {errores} errores, '
      f'{mins:.1f} min, tokens {tok_in:,}/{tok_out:,}, ~USD {costo:.3f}')

openai: lote 1/48
openai: lote 6/48
openai: lote 11/48
openai: lote 16/48
openai: lote 21/48
openai: lote 26/48
openai: lote 31/48
openai: lote 36/48
openai: lote 41/48
openai: lote 46/48

--- openai terminado: 948 etiquetas, 0 errores, 1.9 min, tokens 77,655/8,069, ~USD 0.016


In [3]:
# Celda C7 - Tabla final: precision de los 5 proveedores contra la etiqueta validada
final = pd.read_csv(CLAS / 'piloto_etiquetado_final.csv', keep_default_na=False, na_values=[''])
VALIDAS = {'compra', 'venta', 'neutral'}
PROVS5 = ['deepseek', 'mistral', 'claude', 'kimi', 'openai']

tabla = final[['ticker', 'texto', 'reglas_v2', 'etiqueta_final', 'fuente_etiqueta']].copy()
for p in PROVS5:
    t = pd.read_csv(CLAS / f'clasif_llm_{p}.csv').sort_values('idx')
    tabla[p] = t.etiqueta.values
    tabla[p] = tabla[p].where(tabla[p].isin(VALIDAS))

filas = []
for c in ['reglas_v2'] + PROVS5:
    m = tabla[[c, 'etiqueta_final']].dropna()
    fila = {'clasificador': c,
            'precision_global': round(100 * (m[c] == m.etiqueta_final).mean(), 1)}
    for fuente, nombre in [('consenso_4de4', 'en_oro'), ('mayoria_3de4', 'en_mayoria'),
                           ('pedro_adjudicacion', 'en_dificiles')]:
        sub = tabla[tabla.fuente_etiqueta == fuente][[c, 'etiqueta_final']].dropna()
        fila[nombre] = round(100 * (sub[c] == sub.etiqueta_final).mean(), 1)
    # precision solo en mensajes direccionales (compra/venta reales): lo que importa para B(t)
    dirr = tabla[tabla.etiqueta_final.isin(['compra', 'venta'])][[c, 'etiqueta_final']].dropna()
    fila['en_direccionales'] = round(100 * (dirr[c] == dirr.etiqueta_final).mean(), 1)
    filas.append(fila)

resumen5 = pd.DataFrame(filas).sort_values('precision_global', ascending=False)
resumen5.to_csv(CLAS / 'leaderboard_5_proveedores.csv', index=False)
print(resumen5.to_string(index=False))

clasificador  precision_global  en_oro  en_mayoria  en_dificiles  en_direccionales
      claude              91.5   100.0        88.7          57.7              91.2
        kimi              87.1   100.0        73.4          54.5              85.6
    deepseek              83.7   100.0        74.6          26.0              79.6
     mistral              82.6   100.0        63.3          39.8              81.5
      openai              80.3    94.1        70.6          35.8              75.3
   reglas_v2              49.4    56.0        41.1          35.0              51.1


In [4]:
# Celda C8 - Muestra grande (~15,000 mensajes, 16 subreddits, 2020-2026, semilla 43)
import random, io
import zstandard as zstd

random.seed(43)
import os
DATA = Path(os.environ.get('REDDIT_DATA', '/Users/ppizam/Library/CloudStorage/GoogleDrive-pizacapital@gmail.com/Other computers/My Mac RRG/data/reddit'))
MATRIX = BASE / 'Desarrollo' / 'Metodologia' / 'Matrix'

# universo: los 1,012 tickers con evento
rank = pd.read_csv(MATRIX / 'eventos' / 'ranking_tickers_eventos.csv',
                   keep_default_na=False, na_values=[''])
TICKS = set(rank['ticker'])

# exclusiones de token congeladas del buscador v11
exc_txt = (MATRIX / 'logs' / 'exclusiones_congeladas.txt').read_text()
SIMB_EXCL = set(exc_txt.split('---PALABRAS---')[0].split())
print(f'tickers universo: {len(TICKS)} | tokens excluidos: {len(SIMB_EXCL)}')

RE_CASH = re.compile(r'\$([A-Za-z]{1,5})\b')
RE_TOK = re.compile(r'(?<![A-Za-z$])[A-Z]{2,5}(?![A-Za-z])')

def tickers_en(texto):
    ups = {m.group(1).upper() for m in RE_CASH.finditer(texto)}
    ups |= {t for t in RE_TOK.findall(texto) if t not in SIMB_EXCL}
    return ups & TICKS

SUBS = ['wallstreetbets', 'stocks', 'investing', 'options', 'pennystocks',
        'StockMarket', 'Daytrading', 'Superstonk', 'Shortsqueeze', 'SqueezePlays',
        'SPACs', 'SatoshiStreetBets', 'SecurityAnalysis', 'Vitards',
        'WallStreetbetsELITE', 'Wallstreetbetsnew']
MESES = {a: random.sample(range(1, 13), 2) for a in range(2020, 2026)}
MESES[2026] = random.sample(range(1, 7), 2)
CUOTA, MAX_LINEAS, P_CAP = 40, 200_000, 0.3

muestras = []
for anio, meses in MESES.items():
    for mes in meses:
        for sub in SUBS:
            for tipo in ('submissions', 'comments'):
                ruta = DATA / f'{anio}' / f'{mes:02d}' / f'{sub}_{tipo}.zst'
                if not ruta.exists() or ruta.stat().st_size == 0:
                    continue
                cap = 0
                dctx = zstd.ZstdDecompressor(max_window_size=2**31)
                with open(ruta, 'rb') as f, dctx.stream_reader(f) as r:
                    for n, linea in enumerate(io.TextIOWrapper(r, encoding='utf-8',
                                                               errors='replace')):
                        if n >= MAX_LINEAS or cap >= CUOTA:
                            break
                        try:
                            msg = json.loads(linea)
                        except json.JSONDecodeError:
                            continue
                        if tipo == 'submissions':
                            cuerpo = (msg.get('selftext') or '')
                            if cuerpo in ('[removed]', '[deleted]'):
                                cuerpo = ''
                            texto = ((msg.get('title') or '') + '\n' + cuerpo).strip()[:800]
                        else:
                            texto = (msg.get('body') or '')[:800]
                            if texto in ('[removed]', '[deleted]', ''):
                                continue
                        tks = tickers_en(texto)
                        if not tks or random.random() > P_CAP:
                            continue
                        muestras.append({'anio': anio, 'mes': mes, 'sub': sub,
                                         'tipo': tipo, 'id': msg.get('id', ''),
                                         'created_utc': msg.get('created_utc', ''),
                                         'ticker': sorted(tks)[0], 'n_tickers': len(tks),
                                         'texto': texto})
                        cap += 1
        print(f'{anio}-{mes:02d} listo ({len(muestras):,} acumulados)', flush=True)

mg = pd.DataFrame(muestras).drop_duplicates('id')
# recorte estratificado a ~15,000 por (anio, sub), sin balancear direccion (a proposito)
objetivo_celda = 15000 // mg.groupby(['anio', 'sub']).ngroups
mg15 = (mg.groupby(['anio', 'sub'], group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max(objetivo_celda, 1)), random_state=43),
                 include_groups=True)).reset_index(drop=True)
mg15.to_csv(CLAS / 'muestra_grande.csv', index=False)
print(f'\nmuestra cruda: {len(mg):,} | muestra grande final: {len(mg15):,}')
print('\npor anio:')
print(mg15.groupby('anio').size().to_string())
print('\npor subreddit:')
print(mg15.groupby('sub').size().to_string())

tickers universo: 1012 | tokens excluidos: 129
2020-01 listo (530 acumulados)
2020-05 listo (1,174 acumulados)
2021-12 listo (2,359 acumulados)
2021-03 listo (3,423 acumulados)
2022-08 listo (4,556 acumulados)
2022-06 listo (5,645 acumulados)
2023-11 listo (6,518 acumulados)
2023-02 listo (7,530 acumulados)
2024-08 listo (8,470 acumulados)
2024-10 listo (9,390 acumulados)
2025-08 listo (10,291 acumulados)
2025-10 listo (11,237 acumulados)
2026-01 listo (12,160 acumulados)
2026-05 listo (13,047 acumulados)

muestra cruda: 13,047 | muestra grande final: 11,941

por anio:
anio
2020    1080
2021    2036
2022    2041
2023    1770
2024    1681
2025    1685
2026    1648

por subreddit:
sub
Daytrading             952
SPACs                  539
SatoshiStreetBets      395
SecurityAnalysis       111
Shortsqueeze           852
SqueezePlays           439
StockMarket            971
Superstonk             796
Vitards                411
WallStreetbetsELITE    835
Wallstreetbetsnew      691
investing  

/var/folders/60/tk83x4j51hbb2zy865qqy8j80000gn/T/ipykernel_36098/1224385677.py:79: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max(objetivo_celda, 1)), random_state=43),


In [5]:
# Celda C9 - Etiquetado de la muestra grande con el equipo aprobado (checkpoint por proveedor)
EQUIPO = ['deepseek', 'openai', 'claude']   # baratos primero; claude al final
mg15 = pd.read_csv(CLAS / 'muestra_grande.csv', keep_default_na=False, na_values=[''])

resumen = []
for prov in EQUIPO:
    archivo = CLAS / f'clasif_mg_{prov}.csv'
    hechas = pd.read_csv(archivo) if archivo.exists() else pd.DataFrame(columns=['idx', 'etiqueta'])
    listos = set(hechas['idx'])
    nuevas, tok_in, tok_out, t0 = [], 0, 0, time.time()
    for ini in range(0, len(mg15), LOTE):
        df = mg15.iloc[ini:ini + LOTE]
        if all(ini + j in listos for j in range(len(df))):
            continue
        r = preguntar(prov, prompt_lote(df), system=SYSTEM, max_tokens=2000)
        etiquetas = None
        if not r.get('error'):
            etiquetas = parsear(r['respuesta'], len(df))
            tok_in += r.get('tokens_entrada') or 0
            tok_out += r.get('tokens_salida') or 0
        if etiquetas is None:
            etiquetas = ['error'] * len(df)
        for j, e in enumerate(etiquetas):
            nuevas.append({'idx': ini + j, 'etiqueta': e})
        if (ini // LOTE) % 25 == 0:
            print(f'{prov}: lote {ini // LOTE + 1}/{(len(mg15) - 1) // LOTE + 1}', flush=True)
            pd.concat([hechas, pd.DataFrame(nuevas)]).to_csv(archivo, index=False)
    todas = pd.concat([hechas, pd.DataFrame(nuevas)]).drop_duplicates('idx').sort_values('idx')
    todas.to_csv(archivo, index=False)
    mins = (time.time() - t0) / 60
    errores = int((todas.etiqueta == 'error').sum())
    resumen.append({'proveedor': prov, 'etiquetas': len(todas), 'errores': errores,
                    'tokens_in': tok_in, 'tokens_out': tok_out, 'minutos': round(mins, 1)})
    print(f'--- {prov}: {len(todas)} etiquetas, {errores} errores, {mins:.1f} min')

print()
print(pd.DataFrame(resumen).to_string(index=False))

deepseek: lote 1/598
deepseek: lote 26/598
deepseek: lote 51/598
deepseek: lote 76/598
deepseek: lote 101/598
deepseek: lote 126/598
deepseek: lote 151/598
deepseek: lote 176/598
deepseek: lote 201/598
deepseek: lote 226/598
deepseek: lote 251/598
deepseek: lote 276/598
deepseek: lote 301/598
deepseek: lote 326/598
deepseek: lote 351/598
deepseek: lote 376/598
deepseek: lote 401/598
deepseek: lote 426/598
deepseek: lote 451/598
deepseek: lote 476/598
deepseek: lote 501/598
deepseek: lote 526/598
deepseek: lote 551/598
deepseek: lote 576/598
--- deepseek: 11941 etiquetas, 2020 errores, 146.8 min
openai: lote 1/598
openai: lote 26/598
openai: lote 51/598
openai: lote 76/598
openai: lote 101/598
openai: lote 126/598
[openai] intento 1 fallo (APIConnectionError), reintentando en 2s...
openai: lote 151/598
openai: lote 176/598
openai: lote 201/598
openai: lote 226/598
openai: lote 251/598
openai: lote 276/598
openai: lote 301/598
openai: lote 326/598
openai: lote 351/598
openai: lote 376/59

In [7]:
# Celda C9b - Reparar los lotes fallidos de DeepSeek (modo pensante apagado)
import importlib
import llm_clients
importlib.reload(llm_clients)
from llm_clients import preguntar

prov = 'deepseek'
archivo = CLAS / f'clasif_mg_{prov}.csv'
tabla = pd.read_csv(archivo)
malos = sorted(tabla[tabla.etiqueta == 'error'].idx)
lotes = sorted({i - (i % LOTE) for i in malos})
print(f'{prov}: {len(malos)} etiquetas en error, {len(lotes)} lotes a repetir')

t0 = time.time()
for k, ini in enumerate(lotes):
    df = mg15.iloc[ini:ini + LOTE]
    r = preguntar(prov, prompt_lote(df), system=SYSTEM, max_tokens=2000)
    etiquetas = None if r.get('error') else parsear(r['respuesta'], len(df))
    if etiquetas is None:
        print(f'  lote {ini // LOTE}: sigue fallando')
        continue
    for j, e in enumerate(etiquetas):
        tabla.loc[tabla.idx == ini + j, 'etiqueta'] = e
    if k % 20 == 0:
        tabla.to_csv(archivo, index=False)
        print(f'{k + 1}/{len(lotes)} lotes reparados', flush=True)

tabla.to_csv(archivo, index=False)
quedan = int((tabla.etiqueta == 'error').sum())
print(f'\n{prov} reparado: quedan {quedan} errores | {(time.time() - t0) / 60:.1f} min')

deepseek: 2020 etiquetas en error, 101 lotes a repetir
1/101 lotes reparados
21/101 lotes reparados
41/101 lotes reparados
61/101 lotes reparados
81/101 lotes reparados
101/101 lotes reparados

deepseek reparado: quedan 0 errores | 3.6 min


In [10]:
# Celda C10 - Consolidacion: mayoria 2/3 con desempate a Claude
VALIDAS = {'compra', 'venta', 'neutral'}
et = mg15.copy()
for prov in EQUIPO:
    t = pd.read_csv(CLAS / f'clasif_mg_{prov}.csv').sort_values('idx')
    et[prov] = t.etiqueta.values
    et[prov] = et[prov].where(et[prov].isin(VALIDAS))

def resolver(fila):
    votos = [v for v in [fila['deepseek'], fila['openai'], fila['claude']] if pd.notna(v)]
    if not votos:
        return None, 0, True
    vc = pd.Series(votos).value_counts()
    if vc.iloc[0] >= 2:
        return vc.index[0], int(vc.iloc[0]), False
    # empate a tres bandas: gana claude, con flag
    return fila['claude'], 1, True

res = et.apply(resolver, axis=1)
et['etiqueta_equipo'] = res.str[0]
et['votos'] = res.str[1]
et['flag_empate'] = res.str[2]

et.to_csv(CLAS / 'muestra_grande_etiquetada.csv', index=False)
print(f'muestra etiquetada: {len(et):,} mensajes')
print(f"unanimidad 3/3: {(et.votos == 3).mean():.1%} | mayoria 2/3: {(et.votos == 2).mean():.1%} | "
      f"empates resueltos por claude: {int(et.flag_empate.sum())}")
print('\ndistribucion de la etiqueta del equipo:')
print(et.etiqueta_equipo.value_counts().to_string())
print('\npor tipo de mensaje:')
print(pd.crosstab(et.tipo, et.etiqueta_equipo).to_string())
print('\nguardado: muestra_grande_etiquetada.csv (el conjunto de entrenamiento del fine-tune)')

muestra etiquetada: 11,941 mensajes
unanimidad 3/3: 72.2% | mayoria 2/3: 26.7% | empates resueltos por claude: 127

distribucion de la etiqueta del equipo:
etiqueta_equipo
neutral    6304
compra     4383
venta      1254

por tipo de mensaje:
etiqueta_equipo  compra  neutral  venta
tipo                                   
comments           2121     3334    871
submissions        2262     2970    383

guardado: muestra_grande_etiquetada.csv (el conjunto de entrenamiento del fine-tune)
